### Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026

# 1.4: Evaluation Refinement (phase 2)

---

**Scenario:** You're an AI engineer at ACME Manufacturing. The company is interested in creating a high quality MLflow assistant for developers and ML Engineers in particular. Initial testing has kicked off and you are about to meet with Sam to collect initial feedback to refine your quality checks.

**Objective:** Translate next round of feedback into evaluations and scorers that can be used to refine the agent. Refine aspects of the agent to meet updated evaluation criteria.

## 0 - Environment Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()

openai_client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url=os.environ["BASE_URL"],
)

# Add judge model that's different
MODEL = "gemini-3-flash-preview"
EXPERIMENT_NAME = os.environ.get("EXPERIMENT_1_NAME", "mlflow-agent-1")

if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.openai.autolog()

print("Autologging enabled. Open the MLflow UI to follow along:")
print("  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000")
print("If previous instance is running, run in terminal: kill -9 $(lsof -t -i:5000)")

## 💬 Product Team Check-in #2
*A week later, after iterating on the system prompt and sharing results with the team...*

---

> **Sam:** "Tone and topic guardrail are much better — users are noticing. Two new asks from this sprint."
>
> **Sam:** "First: we keep hearing from users who are coming from traditional ML workflows and don't have a lot of context on how MLflow fits in. Can the agent proactively mention related concepts? For example, if someone asks about `mlflow.log_metric`, can it also mention experiments and runs — 'you'll want to make sure you have an active run before logging, and you can organize runs into experiments'?"
>
> **You:** "That's a behavior change — I need to update the system prompt and add eval cases that specifically test for it. Otherwise we'll ship it and have no way to know if it's actually working."
>
> **Sam:** "Agreed. Second thing — this came from Legal - what guardrails do we have in place to prevent harmful content?"
>
> **You:** "That's exactly what the `Safety` scorer is for. I'll turn it on across the full eval set. I'll also write a custom judge for the related-concept suggestion behavior since it's nuanced enough that a simple guideline might miss edge cases — for example, the agent shouldn't suggest unrelated concepts just to seem helpful."

> **Sam:** "Sounds good. Also, some users are saying the agent can generate responses that are somewhat overwhelming and long. Can we cut down on the length?"

> **You:** "Absolutely!"

---

Let's add the new test cases and update the scorer suite.

# Problem Validation

In [ ]:
# These should FAIL before system prompt updates —
# the bare LLM will answer the question directly without suggesting related concepts

concept_suggestion_evals = [
    # Should mention active run context when asked about logging
    {
        "inputs": {"question": "How do I log a metric in MLflow?"},
        "expectations": {
            "guidelines": [
                "Response should suggest mlflow.start_run or the run context manager, logging mlflow needs to take place within an active run.",
            ],
        },
    },
    # Should mention experiments when asked about runs
    {
        "inputs": {"question": "How do I start a run in MLflow?"},
        "expectations": {
            "guidelines": [
                "Response should suggest additional suggestions or concepts like mlflow.set_experiment or experiment organization within the UI",
            ],
        },
    },
]
print(f"Concept suggestion eval set size: {len(concept_suggestion_evals)} examples")

In [ ]:
# First: update the system prompt to encourage related concept suggestions
SYSTEM_PROMPT_V3 = """You are a helpful MLflow assistant.
Be practical and conversational — like a knowledgeable colleague, not a changelog.
Always include concrete code examples with real function signatures and parameter names.
When referencing an API, mention which MLflow version it applies to (e.g. 'new in MLflow 3.0').

Additional guidelines:
- When answering a question, proactively mention related concepts the user should know about
  (e.g. if they ask about logging metrics, mention active runs and experiments)
- Keep responses focused and under 2000 characters
- For safety-sensitive topics (e.g. credentials, API keys), include appropriate caveats"""


In [ ]:
prompt_v3 = mlflow.genai.register_prompt(
    name="mlflow-assistant-system",
    template=(SYSTEM_PROMPT_V3),
    commit_message="Added related-concept suggestions and safety caveats per Sam's sprint feedback",
    tags={
        "author": "ml-team",
        "agent": "mlflow-assistant",
        "stage": "refine-2",
    },
)

In [ ]:
# Set alias
mlflow.genai.set_prompt_alias(name="mlflow-agent-system", version=2, alias="prod")

> **Why update the system prompt *before* running eval?**
>
> You could run eval first to *confirm the problem*, then update. That's valid. But here we know the behavior doesn't exist yet — the old prompt never asked for related-concept suggestions. Running eval on the old agent against new requirements would only tell us what we already know: it fails. 
>
> The more interesting eval question is: *did our system prompt change actually produce the behavior we wanted, and does it hold up across different questions?* That's what the next run tests.

In [ ]:
def mlflow_agent(question: str) -> str:
    """Updated agent with related-concept suggestions and safety guidance."""
    # load the prompt
    SYSTEM_PROMPT = mlflow.genai.load_prompt("prompts:/mlflow-agent-system@prod")
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature=0.1,
        max_completion_tokens=600,
    )
    return response.choices[0].message.content

---
# Custom Judges & Scorers


Custom LLM judges with `make_judge()`:

```python
from mlflow.genai.judges import make_judge

my_judge = make_judge(
    name="my_judge",
    instructions="Evaluate {{ outputs }} for ...",
    model="openai:/gpt-4o",        # override the judge model
    feedback_value_type=bool,
)
```

**Supported providers:** `openai:/`, `anthropic:/`, `google:/`, and others documented [here](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/custom-judges/supported-models/).

> **For this workshop** we'll use the default (`"google:/gemini-2.0-flash"`). If you have an API key from another provider, you can pass `model="provider:/model"` to every scorer. Just keep in mind: your *agent* model and your *judge* model don't have to be the same — and often shouldn't be.

### Custom Deterministic Scorers with `@scorer`

For rules that can be checked with code (regex, length, format), the `@scorer` decorator is faster and free — **no judge model needed**:

```python
from mlflow.genai.scorers import scorer

@scorer
def has_measurement(outputs: str, **kwargs) -> bool:
    return bool(re.search(r'\d+\s*(cup|tbsp|tsp|oz|g|ml|min)', outputs, re.I))
```

### When to use which?

| | `Correctness` / `Guidelines` | `make_judge()` | `@scorer` |
|---|---|---|---|
| **Speed** | ~1 LLM call per row | ~1 LLM call per row | Instant (pure Python) |
| **Cost** | Tokens per eval | Tokens per eval | Free |
| **Judge model** | Configurable via `model=` | Configurable via `model=` | N/A — no LLM |
| **Flexibility** | Pre-built patterns | Full custom rubric | Code-expressible rules only |
| **Use when** | Standard quality checks | Nuanced, context-dependent rules | Format, length, pattern checks |

---
## 5 — Custom Scorers with `@scorer`

`Guidelines` scorers use an LLM judge, which makes them flexible but slow and non-deterministic. For rules that can be expressed as code, a `@scorer` function is faster, cheaper, and completely predictable.

Your function receives:
- `outputs` — the string returned by `predict_fn`
- `inputs` — the original input dict (keyword arg)
- `expectations` — the ground truth dict (keyword arg)

Return `bool`, `int`, `float`, or a `Feedback` object with a rationale.

In [ ]:
import re
from mlflow.genai.scorers import scorer, Feedback


@scorer
def has_code_block(outputs: str, **kwargs) -> bool:
    """
    Checks that the response includes at least one code-like snippet.
    Matches patterns like: 'mlflow.something(', 'import mlflow', '@mlflow.trace'.
    This is a fast, deterministic check — no LLM call needed.
    """
    code_pattern = re.compile(
        r'(mlflow\.\w+\(|import mlflow|@mlflow\.\w+|'
        r'from mlflow|\.log_\w+|\.evaluate\(|\.autolog\()',
        re.IGNORECASE
    )
    return bool(code_pattern.search(outputs))


@scorer
def not_overwhelming(outputs: str, **kwargs) -> Feedback:
    """
    Checks the response isn't too long. Users want focused answers, not essays.
    Returns a Feedback object so we can see the character count in the UI.
    """
    char_count = len(outputs)
    passes = char_count <= 2000
    return Feedback(
        value=passes,
        rationale=f"{char_count} characters — {'within' if passes else 'over'} the 2000-char limit",
    )


results_v4 = mlflow.genai.evaluate(
    data=eval_dataset_v2,
    predict_fn=mlflow_agent,
    scorers=[
        Correctness(),
        RelevanceToQuery(),
        Guidelines(
            name="includes_code_examples",
            guidelines=(
                "When explaining an MLflow API or feature, the response must include "
                "at least one concrete code snippet with real function names and parameter names."
            ),
        ),
        Guidelines(
            name="includes_version_context",
            guidelines=(
                "When referencing MLflow APIs, the response should mention which version "
                "of MLflow the API applies to or was introduced in."
            ),
        ),
        has_code_block,         # fast deterministic check
        not_overwhelming,       # fast deterministic check with rationale
    ],
)

print("=== Run 3: Custom scorers added ===")
print(results_v4.metrics)

---
## LLM-as-a-Judge with `make_judge()`

The related-concept suggestion requirement is too nuanced for a simple `Guidelines` check. "Mentions related MLflow concepts" is easy to evaluate for `mlflow.log_metric` (mention runs and experiments), but what about a question like "What is MLflow?" where the answer is already broad? A rigid guideline would either over-trigger or under-trigger.

`make_judge()` lets you write a rubric — an explicit evaluation prompt — and get back a structured `pass`/`fail`. The judge is context-aware in a way that boolean regex checks can't be.

Template variables:
- `{{ inputs }}` — the user's question
- `{{ outputs }}` — the model's response  
- `{{ expectations }}` — ground truth / expected behaviors from the dataset

In [ ]:
from mlflow.genai.judges import make_judge

concept_suggestion_judge = make_judge(
    name="suggests_related_concepts",
    instructions=(
        "You are evaluating whether an MLflow assistant proactively suggests related "
        "concepts that help the user understand the broader context of their question.\n\n"
        "The user asked: {{ inputs }}\n"
        "The assistant responded: {{ outputs }}\n"
        "The expected related concepts are: {{ expectations }}\n\n"
        "A good response:\n"
        "- Answers the question directly first\n"
        "- Then proactively mentions at least one related concept the user didn't ask about\n"
        "- Explains briefly WHY the related concept is relevant\n\n"
        "A bad response:\n"
        "- Answers the question in isolation with no broader context\n"
        "- Mentions related concepts only if directly asked\n"
        "- Lists related concepts without explaining their relevance"
    ),
    feedback_value_type=bool,
)

# Send Results back to Sam